# Imports and constants

In [1]:
import re
import os
import ast
import time

K = QuadraticField(-1, 'i')

# Lattice

In [2]:
# Instructions
# The adversary provides a white-box Python implementation of a bilinear multiplication on a free module,
# written in a syntactically restricted normal form where each output update is an explicit sum of scalar multiples of single products f[i]*g[j];
# Note that f and g are read only, h is write only, for writing values in h, the code should be of the form 'h[...] += c*f[...]*g[...]' only
# under these conditions, the full structure tensor of the algebra can be extracted in a single execution.


def extract_operator(filename, funcname, args):
  # Extract code from file
  with open(f"/home/qsdal2/{filename}", "r") as f:
    source = f.read()
  tree = ast.parse(source)
  nodes = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == funcname]
  if len(nodes) != 1:
    return None
  source_code = ast.unparse(nodes[0])

  # Modify source code
  lines = source_code.strip().split('\n')
  newlines = [f'def {funcname}_modified(f, g, h, args):']
  for i in range(1, len(lines)):
    line = lines[i]
    x = re.fullmatch(r'([ \t]*)(h)([ \t]*)(\[.*?\])([ \t]*)(\+=|-=|=|:=)([ \t]*-?[ \t]*((?:\d+|[a-zA-Z_]\w*)[ \t]*\*)?[ \t]*)(f)([ \t]*)(\[.*?\])([ \t]*)(\*)([ \t]*)(g)([ \t]*)(\[.*?\])', line)
    d = None if x is None else {i: x.group(i) for i in range(1, 18)}
    newline = line if x is None else f'{d[1]}{d[2]}{d[3]}{d[11]}{d[17]}{d[4]}{d[5]}{d[6]}{d[7]}{d[9]}{d[10]}{d[11]}{d[12]}{d[13]}{d[14]}{d[15]}{d[16]}{d[17]}'
    newlines.append(newline)
  modified_code = '\n'.join(newlines)
  # print(modified_code)

  # Get the bilinear operator
  namespace = {}
  exec(compile(f'{source}\n\n{modified_code}', "<generated>", "exec"), namespace)
  mul_extracted = namespace[f'{funcname}_modified']
  n = args[0]
  f = [1] * n
  g = [1] * n
  P = [[[0 for k in range(n)] for j in range(n)] for i in range(n)]
  mul_extracted(f, g, P, args)

  return P


def is_associative(P, n):
  for i in range(n):
    for j in range(n):
      for k in range(n):
        for s in range(n):
          if sum(P[i][j][r]*P[r][k][s]-P[j][k][r]*P[i][r][s] for r in range(n)) != 0:
            return False
  return True


def has_identity(P, n, tol=1e-10):
  Ab = matrix(QQ, [[P[i//n][k][i%n] if k < n else 1 if (i//n == i%n) else 0 for k in range(n+1)] for i in range(n*n)])
  N = Ab.right_kernel().basis()
  # print(N)
  if len(N) == 0:
    return False
  for v in N:
    if abs(v[-1]) > tol:
      return True
  return False


def extract_lattice(P, n, mul_type = 'LL'):
  if mul_type == 'LL':
    return [matrix(QQ, [[P[i][k][j] for j in range(n)] for i in range(n)]) for k in range(n)]
  if mul_type == 'LR':
    return [matrix(QQ, [[P[j][k][i] for j in range(n)] for i in range(n)]) for k in range(n)]
  if mul_type == 'RL':
    return [matrix(QQ, [[P[k][i][j] for j in range(n)] for i in range(n)]) for k in range(n)]
  if mul_type == 'RR':
    return [matrix(QQ, [[P[k][j][i] for j in range(n)] for i in range(n)]) for k in range(n)]
  return None


# Lattice Basis
def print_matrix_basis(L, t, n, fname, s1, s2):
  print(f'Basis Matrices for scheme \'{fname}\', Lattice {s1}({s2}) = {s2}0*{s1}0 + ... + {s2}{t-1}*{s1}{t-1}, where')
  X = [str(L[i]).split('\n') for i in range(n)]
  A = [f'{s1}{i} = ' for i in range(t)]
  B = [''.ljust(len(A[i])) for i in range(t)]
  for i in range(n):
    print(f'{'  ' if i < n-1 else ', '}'.join([f'{A[j] if i == 0 else B[j]}{X[i][j]}' for j in range(t)]))


# Lattice Symbolic
def print_matrix_symbolic(L, t, n, fname, s1, s2):
  h = var([f"{s2}{i}" for i in range(t)])
  H = [[str(expand(sum(L[k][i,j]*h[k] for k in range(t)))) for j in range(n)] for i in range(n)]
  m = max(len(H[i//n][i%n]) for i in range(n*n))
  for i in range(n):
    print(f'[{' '.join([H[i][j].rjust(m) for j in range(n)])}]')




# Example of Matrix generation from file
filename = "example.py"
funcname = "mul_NTRU"
n = 4
P = extract_operator(filename, funcname, [n])
L = extract_lattice(P, n)
print_matrix_basis(L, n, n, funcname, 'M_LL', 'h')
print_matrix_symbolic(L, n, n, funcname, 'M_LL', 'h')
R = extract_lattice(P, n, mul_type='RR')
print_matrix_basis(R, n, n, funcname, 'M_RR', 'h')
print_matrix_symbolic(R, n, n, funcname, 'M_RR', 'h')
print(f'Is Associative? => {is_associative(P, n)}')
print(f'Has Identity?   => {has_identity(P, n)}')

Basis Matrices for scheme 'mul_NTRU', Lattice M_LL(h) = h0*M_LL0 + ... + h3*M_LL3, where
M_LL0 = [1 0 0 0]  M_LL1 = [0 1 0 0]  M_LL2 = [0 0 1 0]  M_LL3 = [0 0 0 1]
        [0 1 0 0]          [0 0 1 0]          [0 0 0 1]          [1 0 0 0]
        [0 0 1 0]          [0 0 0 1]          [1 0 0 0]          [0 1 0 0]
        [0 0 0 1],         [1 0 0 0],         [0 1 0 0],         [0 0 1 0]
[h0 h1 h2 h3]
[h3 h0 h1 h2]
[h2 h3 h0 h1]
[h1 h2 h3 h0]
Basis Matrices for scheme 'mul_NTRU', Lattice M_RR(h) = h0*M_RR0 + ... + h3*M_RR3, where
M_RR0 = [1 0 0 0]  M_RR1 = [0 1 0 0]  M_RR2 = [0 0 1 0]  M_RR3 = [0 0 0 1]
        [0 0 0 1]          [1 0 0 0]          [0 1 0 0]          [0 0 1 0]
        [0 0 1 0]          [0 0 0 1]          [1 0 0 0]          [0 1 0 0]
        [0 1 0 0],         [0 0 1 0],         [0 0 0 1],         [1 0 0 0]
[h0 h3 h2 h1]
[h1 h0 h3 h2]
[h2 h1 h0 h3]
[h3 h2 h1 h0]
Is Associative? => True
Has Identity?   => True


# Homomorphism

In [3]:
def scale_up(v):
  L = lcm([lcm(x.real().denominator(), x.imag().denominator()) for x in v])
  v = L*v
  G = gcd([x for x in v])
  v = v/G
  return v


def common_center_basis(A, k, n):
    M = Matrix(QQ, k*n*n, n*n)
    for t in range(k):
        for i in range(n):
            for j in range(n):
                for r in range(n):
                    M[t*n*n + n*i + j, n*r + j] += A[t][i, r]
                    M[t*n*n + n*i + j, n*i + r] -= A[t][r, j]
    B = []
    for b in M.right_kernel().basis():
        B.append(matrix(QQ, n, n, scale_up(b)))
    return B, len(B)


def PD(A, n):
    R.<x> = QQ[]
    cp = A.charpoly(x)
    cpK = cp.change_ring(K)
    F = cpK.factor()
    P = []
    b = []
    for f,e in F:
        N = (f(A) ** e).right_kernel().basis()
        P.extend([scale_up(v) for v in N])
        b.append(e*f.degree())
    return matrix(K, [[P[j][i] for j in range(n)] for i in range(n)]), b, cpK



def combine(P, n, b, s, k, i, p, m, PP):
  if i == k:
    if m == 1:
      return
    # print(b, ' => ', p)
    D = [0] * (s+1)
    for j in range(s):
      D[j+1] = D[j] + b[j]
    ss = [0] * m
    V = [[] for j in range(m)]
    for j in range(s):
      ss[p[j]] += b[j]
      V[p[j]].extend(P[D[j]:D[j+1]])
    DD = [0] * (m+1)
    for j in range(m):
      DD[j+1] = DD[j] + ss[j]
    M = Matrix(K, n, n)
    for j in range(m):
      for jj in range(DD[j],DD[j+1]):
        M[jj,jj] = j
    V = [v for VV in V for v in VV]
    V = Matrix(K, [[P[j][i] for j in range(n)] for i in range(n)])
    P1, b1, F1 = PD(V*M*V.inverse(), n)
    PP.append([P1, b1])
  else:
    m1 = min(k,m+1)
    for x in range(m1):
      p[i] = x
      combine(P, n, b, s, k, i+1, p, max(m,x+1), PP)


def homomorphism(filename, funcname, args, level, do_print):
    n = args[0]
    if level != 2 and level != 3:
        print('invalid level')
        return None
    M = extract_operator(filename, funcname, args)
    L = extract_lattice(M, n, mul_type = 'LL')
    C = extract_lattice(M, n, mul_type = 'RL')
    t = n
    pol = set()
    cnt = 0
    hom = []
    for i in range(t):
        P, b, f = PD(C[i], n)
        key = tuple(f.coefficients(sparse=False))
        if len(b) == 1:
            print(f'C{i} => Only 1 block')
            continue
        if key in pol:
            print(f'C{i} => Repeated characteristic Polynomial')
            continue
        pol.add(key)
        print(f'C{i} => New characteristic Polynomial')
        # print('\n\nNew polynomial => ', f ,'\n\n')
        PP = []
        if level == 2:
            PP = [[P, b]]
        else:
            k = min(6, len(b))
            combine([P.column(i) for i in range(n)], n, b, len(b), k, 1, [0]*k, 1, PP)
        for X in PP:
            hom.append(X)
            if do_print:
                P = X[0]
                b = X[1]
                cnt += 1
                print(f'\nHomomorphism - {cnt} found')
                print(b)
                print(P)
                P_inv = P.inverse()
                L_new = [P_inv * LL * P for LL in L]
                print_matrix_symbolic(L_new, n, n, f'{funcname}_hom{cnt}', 'L\'', 'h')
    print('\n\nAll done!!!')
    return hom

# Testing

In [14]:
# Level-1 = Symbolic
# Level-2 = Primary Decomposition
# Level-3 = Primary Decomposition + Block Combination


start = time.perf_counter()
hom = homomorphism('example.py', 'mul_NTRU', [64], 2, False)
end = time.perf_counter()
print(end - start)

C0 => Only 1 block
C1 => New characteristic Polynomial
C2 => New characteristic Polynomial
C3 => Repeated characteristic Polynomial
C4 => New characteristic Polynomial
C5 => Repeated characteristic Polynomial
C6 => Repeated characteristic Polynomial
C7 => Repeated characteristic Polynomial
C8 => New characteristic Polynomial
C9 => Repeated characteristic Polynomial
C10 => Repeated characteristic Polynomial
C11 => Repeated characteristic Polynomial
C12 => Repeated characteristic Polynomial
C13 => Repeated characteristic Polynomial
C14 => Repeated characteristic Polynomial
C15 => Repeated characteristic Polynomial
C16 => New characteristic Polynomial
C17 => Repeated characteristic Polynomial
C18 => Repeated characteristic Polynomial
C19 => Repeated characteristic Polynomial
C20 => Repeated characteristic Polynomial
C21 => Repeated characteristic Polynomial
C22 => Repeated characteristic Polynomial
C23 => Repeated characteristic Polynomial
C24 => Repeated characteristic Polynomial
C25 => 

In [12]:
print(len(hom))

print(hom[5][0])

7
[ 1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
[ 0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0 